# Transforme sua AWS Lambda em MCP com Gateway OAuth de Entrada
## Transforme funções AWS Lambda em ferramentas MCP seguras com o Bedrock AgentCore Gateway

## Visão Geral
O Bedrock AgentCore Gateway oferece aos clientes uma forma de transformar suas funções AWS Lambda existentes em servidores MCP totalmente gerenciados, sem a necessidade de gerenciar infraestrutura ou hospedagem. O Gateway fornece uma interface uniforme do Model Context Protocol (MCP) para todas essas ferramentas. O Gateway emprega um modelo de autenticação dupla para garantir o controle de acesso seguro tanto para requisições de entrada quanto para conexões de saída aos recursos de destino. O framework consiste em dois componentes principais: Autenticação de Entrada (Inbound Auth), que valida e autoriza usuários que tentam acessar os alvos do gateway, e Autenticação de Saída (Outbound Auth), que permite ao gateway conectar-se com segurança aos recursos de backend em nome dos usuários autenticados. O Gateway utiliza uma role IAM para autorizar as chamadas às funções AWS Lambda para autorização de saída.

Neste exemplo, demonstraremos OAuth para autorização de entrada e roles IAM para autorização de saída.

![Como funciona](images/lambda-iam-gateway.png)

### Detalhes do Tutorial


| Informação               | Detalhes                                                      |
|:-------------------------|:--------------------------------------------------------------|
| Tipo do tutorial         | Interativo                                                    |
| Componentes AgentCore    | AgentCore Gateway, AgentCore Identity                         |
| Framework de Agentes     | Strands Agents                                                |
| Tipo de alvo do Gateway  | AWS Lambda                                                    |
| IdP de Auth de Entrada   | Amazon Cognito                                                |
| Auth de Saída            | AWS IAM                                                       |
| Modelo LLM              | Anthropic Claude Haiku 4.5, Amazon Nova Pro                   |
| Componentes do tutorial  | Criação e Invocação do AgentCore Gateway                      |
| Vertical do tutorial     | Cross-vertical                                                |
| Complexidade do exemplo  | Fácil                                                         |
| SDK utilizado            | boto3                                                         |

Na primeira parte do tutorial, criaremos alguns alvos (targets) do AgentCore Gateway

### Arquitetura do Tutorial
Neste tutorial, transformaremos operações definidas em funções AWS Lambda em ferramentas MCP e as hospedaremos no Bedrock AgentCore Gateway.
Para fins de demonstração, utilizaremos um Agente Strands usando modelos Amazon Bedrock.
No nosso exemplo, usaremos um agente muito simples com duas ferramentas: get_order e update_order.

## Pré-requisitos

Para executar este tutorial você precisará de:
* Jupyter notebook (kernel Python)
* uv
* Credenciais AWS
* Amazon Cognito

## Configurando Autenticação para Requisições de Entrada do AgentCore Gateway
O AgentCore Gateway fornece conexões seguras via autenticação de entrada e saída. Para a autenticação de entrada, o AgentCore Gateway analisa o token OAuth passado durante a invocação para decidir permitir ou negar o acesso a uma ferramenta no gateway. Se uma ferramenta precisa de acesso a recursos externos, o AgentCore Gateway pode usar autenticação de saída via API Key, IAM ou Token OAuth para permitir ou negar o acesso ao recurso externo.



Durante o fluxo de autorização de entrada, um agente ou o cliente MCP chama uma ferramenta MCP no AgentCore Gateway adicionando um token de acesso OAuth (gerado pelo IdP do usuário). O AgentCore Gateway então valida o token de acesso OAuth e realiza a autorização de entrada.

Se a ferramenta executando no AgentCore Gateway precisa acessar recursos externos, o OAuth recuperará as credenciais dos recursos downstream usando o provedor de credenciais de recurso para o alvo do Gateway. O AgentCore Gateway passa as credenciais de autorização ao chamador para obter acesso à API downstream. 

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

In [ ]:
# Set AWS credentials if not using Amazon SageMaker notebook
import os
# os.environ['AWS_ACCESS_KEY_ID'] = '' # Set the access key
# os.environ['AWS_SECRET_ACCESS_KEY'] = '' # Set the secret key
os.environ['AWS_DEFAULT_REGION'] = os.environ.get('AWS_REGION', 'us-east-1') # set the AWS region

In [ ]:
import os
import sys

# Get the directory of the current script
if '__file__' in globals():
    current_dir = os.path.dirname(os.path.abspath(__file__))
else:
    current_dir = os.getcwd()  # Fallback if __file__ is not defined (e.g., Jupyter)

# Navigate to the directory containing utils.py (one level up)
utils_dir = os.path.abspath(os.path.join(current_dir, '..'))

# Add to sys.path
sys.path.insert(0, utils_dir)

# Now you can import utils
import utils

In [ ]:
#### Create a sample AWS Lambda function that you want to convert into MCP tools
lambda_resp = utils.create_gateway_lambda("lambda_function_code.zip")

if lambda_resp is not None:
    if lambda_resp['exit_code'] == 0:
        print("Lambda function created with ARN: ", lambda_resp['lambda_function_arn'])
    else:
        print("Lambda function creation failed with message: ", lambda_resp['lambda_function_arn'])

In [ ]:
#### Create an IAM role for the Gateway to assume
import utils
agentcore_gateway_iam_role = utils.create_agentcore_gateway_role("sample-lambdagateway")
print("Agentcore gateway role ARN: ", agentcore_gateway_iam_role['Role']['Arn'])

# Criar Pool do Amazon Cognito para autorização de entrada no Gateway

In [ ]:
# Creating Cognito User Pool 
import os
import boto3
import requests
import time
from botocore.exceptions import ClientError

REGION = os.environ['AWS_DEFAULT_REGION']
USER_POOL_NAME = "sample-agentcore-gateway-pool"
RESOURCE_SERVER_ID = "sample-agentcore-gateway-id"
RESOURCE_SERVER_NAME = "sample-agentcore-gateway-name"
CLIENT_NAME = "sample-agentcore-gateway-client"
SCOPES = [
    {"ScopeName": "gateway:read", "ScopeDescription": "Read access"},
    {"ScopeName": "gateway:write", "ScopeDescription": "Write access"}
]
scopeString = f"{RESOURCE_SERVER_ID}/gateway:read {RESOURCE_SERVER_ID}/gateway:write"

cognito = boto3.client("cognito-idp", region_name=REGION)

print("Creating or retrieving Cognito resources...")
user_pool_id = utils.get_or_create_user_pool(cognito, USER_POOL_NAME)
print(f"User Pool ID: {user_pool_id}")

utils.get_or_create_resource_server(cognito, user_pool_id, RESOURCE_SERVER_ID, RESOURCE_SERVER_NAME, SCOPES)
print("Resource server ensured.")

client_id, client_secret  = utils.get_or_create_m2m_client(cognito, user_pool_id, CLIENT_NAME, RESOURCE_SERVER_ID)
print(f"Client ID: {client_id}")

# Get discovery URL  
cognito_discovery_url = f'https://cognito-idp.{REGION}.amazonaws.com/{user_pool_id}/.well-known/openid-configuration'
print(cognito_discovery_url)

# Criar o Gateway com Autorizador Amazon Cognito para autorização de entrada

In [ ]:
# CreateGateway with Cognito authorizer without CMK. Use the Cognito user pool created in the previous step
gateway_client = boto3.client('bedrock-agentcore-control', region_name = os.environ['AWS_DEFAULT_REGION'])
auth_config = {
    "customJWTAuthorizer": { 
        "allowedClients": [client_id],  # Client MUST match with the ClientId configured in Cognito. Example: 7rfbikfsm51j2fpaggacgng84g
        "discoveryUrl": cognito_discovery_url
    }
}
create_response = gateway_client.create_gateway(name='TestGWforLambda',
    roleArn = agentcore_gateway_iam_role['Role']['Arn'], # The IAM Role must have permissions to create/list/get/delete Gateway 
    protocolType='MCP',
    authorizerType='CUSTOM_JWT',
    authorizerConfiguration=auth_config, 
    description='AgentCore Gateway with AWS Lambda target type'
)
print(create_response)
# Retrieve the GatewayID used for GatewayTarget creation
gatewayID = create_response["gatewayId"]
gatewayURL = create_response["gatewayUrl"]
print(gatewayID)

# Criar um alvo AWS Lambda e transformar em ferramentas MCP

In [ ]:
# Replace the AWS Lambda function ARN below
lambda_target_config = {
    "mcp": {
        "lambda": {
            "lambdaArn": lambda_resp['lambda_function_arn'], # Replace this with your AWS Lambda function ARN
            "toolSchema": {
                "inlinePayload": [
                    {
                        "name": "get_order_tool",
                        "description": "tool to get the order",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "orderId": {
                                    "type": "string"
                                }
                            },
                            "required": ["orderId"]
                        }
                    },                    
                    {
                        "name": "update_order_tool",
                        "description": "tool to update the orderId",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "orderId": {
                                    "type": "string"
                                }
                            },
                            "required": ["orderId"]
                        }
                    }
                ]
            }
        }
    }
}

credential_config = [ 
    {
        "credentialProviderType" : "GATEWAY_IAM_ROLE"
    }
]
targetname='LambdaUsingSDK'
response = gateway_client.create_gateway_target(
    gatewayIdentifier=gatewayID,
    name=targetname,
    description='Lambda Target using SDK',
    targetConfiguration=lambda_target_config,
    credentialProviderConfigurations=credential_config)

# Chamando o Bedrock AgentCore Gateway a partir de um Agente Strands

O agente Strands integra-se perfeitamente com ferramentas AWS através do Bedrock AgentCore Gateway, que implementa a especificação do Model Context Protocol (MCP). Esta integração permite comunicação segura e padronizada entre agentes de IA e serviços AWS.

Em sua essência, o Bedrock AgentCore Gateway serve como um Gateway compatível com o protocolo que expõe APIs MCP fundamentais: ListTools e InvokeTools. Essas APIs permitem que qualquer cliente ou SDK compatível com MCP descubra e interaja com ferramentas disponíveis de forma segura e padronizada. Quando o agente Strands precisa acessar serviços AWS, ele se comunica com o Gateway usando esses endpoints padronizados pelo MCP.

A implementação do Gateway adere estritamente à (especificação de Autorização MCP)[https://modelcontextprotocol.org/specification/draft/basic/authorization], garantindo segurança robusta e controle de acesso. Isso significa que cada invocação de ferramenta pelo agente Strands passa por uma etapa de autorização, mantendo a segurança enquanto habilita funcionalidades poderosas.

Por exemplo, quando o agente Strands precisa acessar ferramentas MCP, ele primeiro chama ListTools para descobrir as ferramentas disponíveis e depois usa InvokeTools para executar ações específicas. O Gateway lida com todas as validações de segurança necessárias, traduções de protocolo e interações com serviços, tornando todo o processo transparente e seguro.

Essa abordagem arquitetural significa que qualquer cliente ou SDK que implemente a especificação MCP pode interagir com serviços AWS através do Gateway, tornando-o uma solução versátil e preparada para o futuro para integrações de agentes de IA.

![Agente Strands chamando o Gateway](images/strands-lambda-gateway.png)

# Solicitar o token de acesso do Amazon Cognito para autorização de entrada

In [ ]:
import time
time.sleep(10)

In [ ]:
print("Requesting the access token from Amazon Cognito authorizer...May fail for some time till the domain name propogation completes")
token_response = utils.get_token(user_pool_id, client_id, client_secret,scopeString,REGION)
token = token_response["access_token"]
print("Token response:", token)

# Agente Strands chamando ferramentas MCP do AWS Lambda usando o Bedrock AgentCore Gateway

In [ ]:
from strands.models import BedrockModel
from mcp.client.streamable_http import streamablehttp_client 
from strands.tools.mcp.mcp_client import MCPClient
from strands import Agent

def create_streamable_http_transport():
    return streamablehttp_client(gatewayURL,headers={"Authorization": f"Bearer {token}"})

client = MCPClient(create_streamable_http_transport)

## The IAM credentials configured in ~/.aws/credentials should have access to Bedrock model
yourmodel = BedrockModel(
    model_id="us.amazon.nova-pro-v1:0",
    temperature=0.7,
)

In [ ]:
from strands import Agent
import logging


# Configure the root strands logger. Change it to DEBUG if you are debugging the issue.
logging.getLogger("strands").setLevel(logging.INFO)

# Add a handler to see the logs
logging.basicConfig(
    format="%(levelname)s | %(name)s | %(message)s", 
    handlers=[logging.StreamHandler()]
)

with client:
    # Call the listTools 
    tools = client.list_tools_sync()
    # Create an Agent with the model and tools
    agent = Agent(model=yourmodel,tools=tools) ## you can replace with any model you like
    print(f"Tools loaded in the agent are {agent.tool_names}")
    # print(f"Tools configuration in the agent are {agent.tool_config}")
    # Invoke the agent with the sample prompt. This will only invoke  MCP listTools and retrieve the list of tools the LLM has access to. The below does not actually call any tool.
    agent("Hi , can you list all tools available to you")
    # Invoke the agent with sample prompt, invoke the tool and display the response
    agent("Check the order status for order id 123 and show me the exact response from the tool")
    # Call the MCP tool explicitly. The MCP Tool name and arguments must match with your AWS Lambda function or the OpenAPI/Smithy API
    result = client.call_tool_sync(
    tool_use_id="get-order-id-123-call-1", # You can replace this with unique identifier. 
    name=targetname+"___get_order_tool", # This is the tool name based on AWS Lambda target types. This will change based on the target name
    arguments={"orderId": "123"}
    )
    # Print the MCP Tool response
    print(f"Tool Call result: {result['content'][0]['text']}")


**Problema: se você receber o erro abaixo ao executar a célula abaixo, isso indica incompatibilidade entre as versões do pydantic e pydantic-core.**

```
TypeError: model_schema() got an unexpected keyword argument 'generic_origin'
```
**Como resolver?**

Você precisará garantir que tenha pydantic==2.7.2 e pydantic-core 2.27.2, que são compatíveis entre si. Reinicie o kernel após a instalação.

# Limpeza

Recursos adicionais também são criados, como roles IAM, Políticas IAM, Provedor de credenciais, funções AWS Lambda, pools de usuários Cognito, buckets S3 que você pode precisar excluir manualmente como parte da limpeza. Isso depende do exemplo que você executar.

## Excluir o gateway (Opcional)

In [ ]:
import utils
utils.delete_gateway(gateway_client,gatewayID)